In [19]:
from VinUniData import DataPipeline

DATA_DIR = ''
TABLE_RELATIONSHIPS = {
    "orders":            {"customer_id": "customers"},
    "order_items":       {"order_id": "orders", "product_id": "products"},
    "payments":          {"order_id": "orders"},
    "shipments":         {"order_id": "orders"},
    "returns":           {"order_id": "orders", "product_id": "products"},
    "reviews":           {"customer_id": "customers", "product_id": "products", "order_id": "orders"},
    "inventory":         {"product_id": "products"},
    "web_traffic":       {}   
    # geography, promotions ít phụ thuộc, có thể kiểm tra nếu cần
}

TABLE_MAP = {
    # Master
    "products":    "master/products.csv",
    "customers":   "master/customers.csv",
    "promotions":  "master/promotions.csv",
    "geography":   "master/geography.csv",
    # Transaction
    "orders":      "transaction/orders.csv",
    "order_items": "transaction/order_items.csv",
    "payments":    "transaction/payments.csv",
    "shipments":   "transaction/shipments.csv",
    "returns":     "transaction/returns.csv",
    "reviews":     "transaction/reviews.csv",
    # Analytical
    "sales":       "analytical/sales.csv",
    # Operational
    "inventory":   "operational/inventory.csv",
    "web_traffic": "operational/web_traffic.csv",
}

DATE_COLS = {
    "orders":      ["order_date"],
    "customers": ["signup_date"],
    "shipments":   ["ship_date", "delivery_date"],
    "returns":     ["return_date"],
    "sales":       ["Date"],
    "inventory":   ["snapshot_date"],
    "web_traffic": ["date"],
    "promotions": ["start_date", "end_date"],
    "reviews": ["review_date"],
}

# Cột cần kiểm tra outlier
OUTLIER_COLS = {
    "order_items": ["quantity", "unit_price"],
    "payments":    ["amount"],
    "sales":       ["revenue"],
    "web_traffic": ["visits", "page_views"],
}

In [20]:
pipeline = DataPipeline(DATA_DIR, TABLE_MAP, TABLE_RELATIONSHIPS)

[LOAD] products: 2412 rows x 8 cols
[LOAD] customers: 121930 rows x 7 cols
[LOAD] promotions: 50 rows x 10 cols
[LOAD] geography: 39948 rows x 4 cols
[LOAD] orders: 646945 rows x 8 cols
[LOAD] order_items: 714669 rows x 7 cols


/Users/quannguyen/Documents/Datathon-VinUni/datathon-2026-round-1/VinUniData.py:20: DtypeWarning: Columns (0: promo_id_2) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(file_path)


[LOAD] payments: 646945 rows x 4 cols
[LOAD] shipments: 566067 rows x 4 cols
[LOAD] returns: 39939 rows x 7 cols
[LOAD] reviews: 113551 rows x 7 cols
[LOAD] sales: 3833 rows x 3 cols
[LOAD] inventory: 60247 rows x 17 cols
[LOAD] web_traffic: 3652 rows x 7 cols


# Data structure and Data types

In [21]:
pipeline.check_structure()


===== STRUCTURE & DATA TYPES =====

--- products ---
Shape: (2412, 8)
Data types distribution:
  str: 5 columns
  float64: 2 columns
  int64: 1 columns
  product_id: int64
  product_name: str
  category: str
  segment: str
  size: str
  color: str
  price: float64
  cogs: float64

--- customers ---
Shape: (121930, 7)
Data types distribution:
  str: 5 columns
  int64: 2 columns
  customer_id: int64
  zip: int64
  city: str
  signup_date: str
  gender: str
  age_group: str
  acquisition_channel: str

--- promotions ---
Shape: (50, 10)
Data types distribution:
  str: 7 columns
  int64: 2 columns
  float64: 1 columns
  promo_id: str
  promo_name: str
  promo_type: str
  discount_value: float64
  start_date: str
  end_date: str
  applicable_category: str
  promo_channel: str
  stackable_flag: int64
  min_order_value: int64

--- geography ---
Shape: (39948, 4)
Data types distribution:
  str: 3 columns
  int64: 1 columns
  zip: int64
  city: str
  region: str
  district: str

--- orders ---


# DATE DATATYPE NORMALIZATION

In [22]:
pipeline.standardize_dates_and_numbers(DATE_COLS)


===== DATE STANDARDIZATION =====
  [DATE] orders.order_date: range 2012-07-04 to 2022-12-31
  [DATE] customers.signup_date: range 2012-01-17 to 2022-12-31
  [DATE] shipments.ship_date: range 2012-07-04 to 2022-12-29
  [DATE] shipments.delivery_date: range 2012-07-06 to 2022-12-31
  [DATE] returns.return_date: range 2012-07-11 to 2022-12-31
  [DATE] sales.Date: range 2012-07-04 to 2022-12-31
  [DATE] inventory.snapshot_date: range 2012-07-31 to 2022-12-31
  [DATE] web_traffic.date: range 2013-01-01 to 2022-12-31
  [DATE] promotions.start_date: range 2013-01-31 to 2022-11-18
  [DATE] promotions.end_date: range 2013-03-01 to 2022-12-31
  [DATE] reviews.review_date: range 2012-07-10 to 2022-12-31


# DATA REFERENCE KEY EVALUATION

In [23]:
pipeline.check_referential_integrity()


===== REFERENTIAL INTEGRITY =====
  orders.customer_id -> customers: 0/646945 orphan keys (0.00%)
  order_items.order_id -> orders: 0/714669 orphan keys (0.00%)
  order_items.product_id -> products: 0/714669 orphan keys (0.00%)
  payments.order_id -> orders: 0/646945 orphan keys (0.00%)
  shipments.order_id -> orders: 0/566067 orphan keys (0.00%)
  returns.order_id -> orders: 0/39939 orphan keys (0.00%)
  returns.product_id -> products: 0/39939 orphan keys (0.00%)
  reviews.customer_id -> customers: 0/113551 orphan keys (0.00%)
  reviews.product_id -> products: 0/113551 orphan keys (0.00%)
  reviews.order_id -> orders: 0/113551 orphan keys (0.00%)
  inventory.product_id -> products: 0/60247 orphan keys (0.00%)


In [24]:
pipeline.count_missing()


===== MISSING VALUES =====
  products: no missing values
  customers: no missing values

--- promotions ---
             column  missing_count  missing_pct
applicable_category             40         80.0
  geography: no missing values
  orders: no missing values

--- order_items ---
    column  missing_count  missing_pct
promo_id_2         714463    99.971175
  promo_id         438353    61.336507
  payments: no missing values
  shipments: no missing values
  returns: no missing values
  reviews: no missing values
  sales: no missing values
  inventory: no missing values
  web_traffic: no missing values
